# DS-2002 Project 2: ClassicModels Data Lakehouse (Medallion Architecture)

This notebook implements a full Data Lakehouse for the ClassicModels database using:

### Source Systems
- MySQL (SQL)
- MongoDB (NoSQL)
- JSON Streaming (Files)

### Architecture
- Bronze — raw streaming fact data  
- Silver — enriched fact table with surrogate keys  
- Gold — aggregated OLAP fact table  

## Project Overview

This project implements a Dimensional Data Lakehouse using the ClassicModels dataset.

It demonstrates:

##### Integration of multiple source systems

- MySQL (orders, customers, employees)

- MongoDB (products)

- JSON streaming files (fact table)

##### Batch and streaming

##### Bronze > Silver > Gold Medallion architecture

##### A dimensional model with Date + Customer + Employee + Product + Fact

##### Business value queries on the Gold layer

#### Import the Necessary Libraries

In [1]:
import findspark
findspark.init()
print(findspark.find())

import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])

import sys
import json
import time
import pymongo
import certifi
import shutil
import numpy as pd

import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark
JAVA_HOME = /Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home


#### Start Spark Session

In [2]:
spark = (
    SparkSession.builder
        .appName("ClassicModels Lakehouse (Medallion Architecture)")
        .master("local[*]")
        .config(
            "spark.jars.packages",
            "com.mysql:mysql-connector-j:9.1.0,"
            "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1"
        )
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.streaming.schemaInference", "true")
        .config("spark.streaming.stopGracefullyOnShutdown", "true")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

26/05/07 15:24:31 WARN Utils: Your hostname, Tianyaos-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.41.1.13 instead (on interface en0)
26/05/07 15:24:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/tianyao/.ivy2/cache
The jars for the packages stored in: /Users/tianyao/.ivy2/jars
com.mysql#mysql-connector-j added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-60ccc1c0-4cd3-46b9-bae6-851ba077fad3;1.0
	confs: [default]
	found com.mysql#mysql-connector-j;9.1.0 in central


:: loading settings :: url = jar:file:/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.google.protobuf#protobuf-java;4.26.1 in central
	found org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 in central
	found org.mongodb#mongodb-driver-sync;4.0.5 in central
	found org.mongodb#bson;4.0.5 in central
	found org.mongodb#mongodb-driver-core;4.0.5 in central
:: resolution report :: resolve 130ms :: artifacts dl 3ms
	:: modules in use:
	com.google.protobuf#protobuf-java;4.26.1 from central in [default]
	com.mysql#mysql-connector-j;9.1.0 from central in [default]
	org.mongodb#bson;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-core;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-sync;4.0.5 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------------------

#### Utility functions and configuration

In [3]:
def get_file_info(path: str):
    '''Return a DataFrame with file name, size, and last modification time for all files in a directory.'''
    file_sizes = []
    modification_times = []

    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])

    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))

    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    '''Block until a streaming query has processed at least min_batches.'''
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
    print(f"The stream has processed {len(query.recentProgress)} batches")


def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    if os.path.exists(path):
        shutil.rmtree(path)

#### MySQL + MongoDB Connection Configuration

In [4]:
# MySQL connection
jdbc_url = "jdbc:mysql://localhost:3306/classicmodels"

mysql_props = {
    "user": "root",
    "password": "NewPasswordHere",   # <-- replace with your actual password
    "driver": "com.mysql.cj.jdbc.Driver"
}

# MongoDB connection (local)
mongo_uri = "mongodb://localhost:27017/"
mongo_db  = "classicmodels_mongo"
mongo_products_collection = "products"

#### Lakehouse directory structure

In [5]:
warehouse_dir = os.path.abspath("spark-warehouse")
db_name = "classicmodels_dlh"

base_dir = os.path.join(os.getcwd(), "data")
orders_stream_dir = os.path.join(base_dir, "orders_stream")

database_dir = os.path.join(warehouse_dir, f"{db_name}.db")

def dim_dir(name):  return os.path.join(database_dir, f"dim_{name}")
def fact_dir(name): return os.path.join(database_dir, f"fact_{name}")

orders_bronze_dir = os.path.join(fact_dir("order_lines"), "bronze")
orders_silver_dir = os.path.join(fact_dir("order_lines"), "silver")
orders_gold_dir   = os.path.join(fact_dir("order_lines"), "gold")

print("warehouse_dir:", warehouse_dir)
print("database_dir:", database_dir)

# Reset Spark SQL database and Lakehouse directories

remove_directory_tree(database_dir)
spark.sql(f"DROP DATABASE IF EXISTS {db_name} CASCADE")
spark.sql(f"CREATE DATABASE {db_name}")
spark.sql(f"USE {db_name}")

warehouse_dir: /Users/tianyao/Desktop/DS-2002-main copy/Projects/spark-warehouse
database_dir: /Users/tianyao/Desktop/DS-2002-main copy/Projects/spark-warehouse/classicmodels_dlh.db


DataFrame[]

## Batch Dimensions (ClassicModels)
- Date (MySQL)
- Customer (MySQL)
- Employees (MySQL)
- Products (MongoDB)
- Fact Table (MySQL)

#### Load Dates (MySQL)

In [6]:
# Date dimension
# Source: orders.orderDate
# Keys and attributes: date_key, year, quarter, month, day

df_dates = (
    spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("query", "SELECT orderDate AS date FROM orders")
        .options(**mysql_props)
        .load()
        .distinct()
        .withColumn("date_key", date_format("date", "yyyyMMdd").cast("int"))
        .withColumn("year", year("date"))
        .withColumn("quarter", quarter("date"))
        .withColumn("month", month("date"))
        .withColumn("day", dayofmonth("date"))
)

df_dates.write.mode("overwrite").parquet(dim_dir("date"))

#### Load Customers (MySQL)

In [7]:
# Customer dimension
# Source: customers table

df_customers = (
    spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "customers")
        .options(**mysql_props)
        .load()
        .select(
            col("customerNumber").alias("customer_key"),
            "customerName",
            "contactLastName",
            "contactFirstName",
            "phone",
            "city",
            "country",
            "salesRepEmployeeNumber"
        )
)

df_customers.write.mode("overwrite").parquet(dim_dir("customer"))

#### Load Employee (MySQL)

In [8]:
# Employee dimension
# Source: employees table

df_employees = (
    spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "employees")
        .options(**mysql_props)
        .load()
        .select(
            col("employeeNumber").alias("employee_key"),
            "lastName",
            "firstName",
            "extension",
            "email",
            "officeCode",
            "reportsTo",
            "jobTitle"
        )
)

df_employees.write.mode("overwrite").parquet(dim_dir("employee"))

#### Load Product from MongoDB (NoSQL)

In [9]:
# Product dimension 
# Source: MongoDB products (NoSQL)

df_products_raw = (
    spark.read.format("com.mongodb.spark.sql.DefaultSource")
        .option("uri", mongo_uri)
        .option("database", mongo_db)
        .option("collection", mongo_products_collection)
        .load()
)

df_products = (
    df_products_raw
        .drop("_id")
        .select(
            col("productCode").alias("product_key"),
            "productName",
            "productLine",
            "productScale",
            "productVendor",
            "quantityInStock",
            "buyPrice",
            "MSRP"
        )
)

df_products.write.mode("overwrite").parquet(dim_dir("product"))

In [10]:
df_products_raw.printSchema()
df_products_raw.show(5, truncate=False)

root
 |-- MSRP: double (nullable = true)
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- buyPrice: double (nullable = true)
 |-- productCode: string (nullable = true)
 |-- productDescription: string (nullable = true)
 |-- productLine: string (nullable = true)
 |-- productName: string (nullable = true)
 |-- productScale: string (nullable = true)
 |-- productVendor: string (nullable = true)
 |-- quantityInStock: integer (nullable = true)

+------+--------------------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
# Join orders and orderdetails to create order line fact rows.

df_orders = (
    spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "orders")
        .options(**mysql_props)
        .load()
)

df_orderdetails = (
    spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "orderdetails")
        .options(**mysql_props)
        .load()
)

df_fact = (
    df_orderdetails.alias("od")
        .join(df_orders.alias("o"), "orderNumber")
        .select(
            col("orderNumber").cast("long"),
            col("orderLineNumber").cast("int"),
            col("quantityOrdered").cast("int"),
            col("priceEach").cast("double"),
            (col("quantityOrdered") * col("priceEach")).alias("extendedAmount"),
            date_format("orderDate", "yyyyMMdd").cast("int").alias("order_date_key"),
            col("customerNumber").cast("int").alias("customer_key"),
            col("productCode").alias("product_key")
        )
)

#### Split the Fact Table into 3 Streaming Intervals

In [12]:
remove_directory_tree(orders_stream_dir)
os.makedirs(orders_stream_dir, exist_ok=True)

pdf_fact = df_fact.toPandas().sample(frac=1, random_state=42)
splits = np.array_split(pdf_fact, 3)

for i, part in enumerate(splits, start=1):
    part.to_json(
        os.path.join(orders_stream_dir, f"orders_part{i}.json"),
        orient="records"
    )

get_file_info(orders_stream_dir)


/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


,name,size,modification_time
0,orders_part1.json,176068,2026-05-07 19:24:46.574930191
1,orders_part2.json,176073,2026-05-07 19:24:46.575816631
2,orders_part3.json,175900,2026-05-07 19:24:46.576701880


## Bronze Layer: Streaming Project 1 Fact Table

The Bronze layer ingests the raw Project 1 fact table as streaming JSON files.
This layer stores the data exactly as received, without enforcing types, constraints, or business rules.
It serves as the immutable, append‑only record of the incoming fact stream.

#### Schema (raw Bronze fact_order_lines):

- orderNumber
- orderLineNumber
- quantityOrdered
- priceEach
- extendedAmount
- order_date_key
- customer_key
- product_key
- receipt_time (Bronze metadata)
- source_file (Bronze metadata)
  
##### Notes:
employee_key is intentionally not present in Bronze

In Project 1, the employee relationship comes from
customers.salesRepEmployeeNumber > employees.employeeNumber

This join is performed in the Silver layer, not Bronze

Bronze stores the fact table exactly as exported, matching the Project 1 schema

In [13]:
# Read streaming JSON files as they arrive and write raw Parquet.

remove_directory_tree(orders_bronze_dir)
os.makedirs(orders_bronze_dir, exist_ok=True)

bronze_checkpoint = os.path.join(orders_bronze_dir, "_checkpoint")

df_orders_bronze = (
    spark.readStream
        .option("maxFilesPerTrigger", 1)
        .option("multiLine", "true")
        .json(orders_stream_dir)
)

bronze_query = (
    df_orders_bronze
        .withColumn("receipt_time", current_timestamp())
        .withColumn("source_file", input_file_name())
        .writeStream
        .format("parquet")
        .outputMode("append")
        .option("checkpointLocation", bronze_checkpoint)
        .start(orders_bronze_dir)
)

wait_until_stream_is_ready(bronze_query, min_batches=3)
bronze_query.stop()

26/05/07 15:24:50 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


The stream has processed 3 batches


## Silver Layer: Cleaned Fact Table

The Silver layer reads the Bronze parquet files and applies:

- Type enforcement

- Business‑rule cleaning

- Dimensional enrichment (Date, Customer, Employee, Product)

The Silver fact table keeps all business columns from Bronze:

#### Schema (Silver fact_order_lines)

- orderNumber
- orderLineNumber
- quantityOrdered
- priceEach
- extendedAmount
- order_date_key
- customer_key
- product_key
- receipt_time (Bronze metadata)
- source_file (Bronze metadata)
- Important Correction
  
employee_key is not present in Silver, because it does not exist in Bronze

Instead, Silver derives employee_key by joining:
customers.salesRepEmployeeNumber > employees.employeeNumber

In [16]:
# Clean types and join with all dimensions (Date, Customer, Employee, Product).

df_bronze_static = spark.read.parquet(orders_bronze_dir)

df_silver_fact = (
    df_bronze_static
        .select(
            col("orderNumber").cast("long"),
            col("orderLineNumber").cast("int"),
            col("quantityOrdered").cast("int"),
            col("priceEach").cast("double"),
            col("extendedAmount").cast("double"),
            col("order_date_key").cast("int"),
            col("customer_key").cast("int"),
            col("product_key"),
            "receipt_time",
            "source_file"
        )
)

dim_date      = spark.read.parquet(dim_dir("date"))
dim_customer  = spark.read.parquet(dim_dir("customer"))
dim_employee  = spark.read.parquet(dim_dir("employee"))
dim_product   = spark.read.parquet(dim_dir("product"))

df_silver_enriched = (
    df_silver_fact.alias("f")
        .join(dim_customer.alias("c"), "customer_key", "left")
        .join(dim_employee.alias("e"),
              col("c.salesRepEmployeeNumber") == col("e.employee_key"),
              "left")
        .join(dim_product.alias("p"), "product_key", "left")
        .join(dim_date.alias("d"), col("f.order_date_key") == col("d.date_key"), "left")
)

df_silver_enriched.write.mode("overwrite").parquet(orders_silver_dir)

26/05/07 15:28:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Gold Layer: Aggregated OLAP Fact

The Gold layer computes business‑ready OLAP metrics from the cleaned and enriched Silver fact table.

This layer represents the final analytical fact table used for reporting, dashboards, and business intelligence.

#### Metrics:
- total_sales = sum(quantityOrdered * priceEach)
- total_quantity = sum(quantityOrdered)
- order_count = count distinct orderNumber

#### Grouped by:
- customer_key
- product_key
- order_date_key

#### Gold Output

The Gold fact table contains:

- Fully aggregated OLAP metrics
- Conformed surrogate keys
- One row per customer–product–date combination

#### This dataset is optimized for:

- Revenue analysis
- Product performance
- Customer purchasing behavior
- Time‑series reporting

In [17]:
# Aggregate Silver fact into a star-schema style fact table for reporting.

df_silver = spark.read.parquet(orders_silver_dir)
df_silver.createOrReplaceTempView("fact_order_lines_silver")

df_gold = spark.sql("""
SELECT
    customer_key,
    product_key,
    order_date_key,
    SUM(quantityOrdered * priceEach) AS total_sales,
    SUM(quantityOrdered) AS total_quantity,
    COUNT(DISTINCT orderNumber) AS order_count
FROM fact_order_lines_silver
GROUP BY customer_key, product_key, order_date_key
""")

df_gold.write.mode("overwrite").parquet(orders_gold_dir)

## Example Queries on Gold Layer

#### Register Gold and dimensions as views

In [18]:
df_gold = spark.read.parquet(orders_gold_dir)
df_gold.createOrReplaceTempView("fact_order_lines_gold")

dim_customer.createOrReplaceTempView("dim_customer")
dim_product.createOrReplaceTempView("dim_product")
dim_date.createOrReplaceTempView("dim_date")

#### Sort Top Customers By Revenue

In [19]:
top_customers = spark.sql("""
SELECT
    c.customerName,
    SUM(g.total_sales) AS revenue
FROM fact_order_lines_gold g
JOIN dim_customer c USING (customer_key)
GROUP BY c.customerName
ORDER BY revenue DESC
LIMIT 10
""")

top_customers.toPandas()

,customerName,revenue
0,Euro+ Shopping Channel,820689.54
1,Mini Gifts Distributors Ltd.,591827.34
2,"Australian Collectors, Co.",180585.07
3,Muscle Machine Inc,177913.95
4,La Rochelle Gifts,158573.12
5,"Dragon Souveniers, Ltd.",156251.03
6,"Down Under Souveniers, Inc",154622.08
7,Land of Toys Inc.,149085.15
8,"AV Stores, Co.",148410.09
9,The Sharp Gifts Warehouse,143536.27


#### Sort Top products by quantity sold

In [20]:
top_products = spark.sql("""
SELECT
    p.productName,
    SUM(g.total_quantity) AS units_sold
FROM fact_order_lines_gold g
JOIN dim_product p USING (product_key)
GROUP BY p.productName
ORDER BY units_sold DESC
LIMIT 10
""")

top_products.toPandas()

,productName,units_sold
0,1992 Ferrari 360 Spider red,1808
1,1937 Lincoln Berline,1111
2,American Airlines: MD-11S,1085
3,1941 Chevrolet Special Deluxe Cabriolet,1076
4,1930 Buick Marquette Phaeton,1074
5,1940s Ford truck,1061
6,1969 Harley Davidson Ultimate Chopper,1057
7,1957 Chevy Pickup,1056
8,1964 Mercedes Tour Bus,1053
9,1956 Porsche 356A Coupe,1052


#### Monthly revenue trend

In [21]:
monthly_revenue = spark.sql("""
SELECT
    d.year,
    d.month,
    SUM(g.total_sales) AS revenue
FROM fact_order_lines_gold g
JOIN dim_date d ON g.order_date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month
""")

monthly_revenue.toPandas()

,year,month,revenue
0,2003,1,116692.77
1,2003,2,128403.64
2,2003,3,160517.14
3,2003,4,185848.59
4,2003,5,179435.55
5,2003,6,150470.77
6,2003,7,201940.36
7,2003,8,178257.11
8,2003,9,236697.85
9,2003,10,514336.21


In [22]:
spark.stop()